# Préparation des entreprises RGE — Val-d'Oise

Ce notebook prépare le fichier national des entreprises RGE pour l'analyse du projet **Rénovation énergétique — Val-d'Oise**.

### Objectifs
- filtrer correctement le département 95 ;
- harmoniser les noms de communes avec le référentiel DPE propre ;
- contrôler les types, doublons et valeurs manquantes ;
- conserver les colonnes utiles à l'analyse ;
- exporter un fichier RGE propre et reproductible.

### Résultats attendus
- **1 607 lignes**
- **553 entreprises RGE distinctes**
- **102 communes**
- **0 commune non reconnue**

## 1. Import de la bibliothèque

In [1]:
import pandas as pd

## 2. Chargement du fichier RGE national

In [2]:
df_rge = pd.read_csv(
    "../Data/Input/liste_des_entreprises_rge.csv"
)

In [3]:
df_rge.head()

,siret,nom_entreprise,adresse,code_postal,commune,latitude,longitude,telephone,email,site_internet,code_qualification,nom_qualification,url_qualification,nom_certificat,domaine,meta_domaine,organisme,particulier,lien_date_debut,lien_date_fin
0,82454695600023,EL ARCHITECTURE,4 RUE D'HALLENNES,59320,ENGLOS,50.622205,2.957894,06 37 51 96 32,e.lannoyarchitecture@gmail.com,NaN,2,Architecte sur le site faire.fr (2),https://ws-api.architectes.org/faire/v2/attest...,Tableau de l´Ordre,Architecte,Rénovation globale,cnoa,1,2019-10-16,2099-01-01
1,48774909500021,SCOP-SARL ENZO & ROSSO,113 BOULEVARD DE LAMASQUERE,31600,MURET,43.469039,1.310118,05 34 46 19 48,contact@enzo-rosso.fr,http://www.enzo-rosso.fr,2,Architecte sur le site faire.fr (2),https://ws-api.architectes.org/faire/v2/attest...,Tableau de l´Ordre,Architecte,Rénovation globale,cnoa,1,2019-06-25,2099-01-01
2,89804121500010,ONE ART STUDIO,50 RUE BLOMET,75015,PARIS,48.842498,2.305153,06 22 23 80 34,oneartstudio@outlook.fr,NaN,2,Architecte sur le site faire.fr (2),https://ws-api.architectes.org/faire/v2/attest...,Tableau de l´Ordre,Architecte,Rénovation globale,cnoa,1,2021-06-29,2099-01-01
3,44781964000015,CABINET GUINARD ARCHITECTE,18 AVENUE D'OSSAU,64110,JURANCON,43.290674,-0.391767,05 59 06 19 47,jpgarchi@wanadoo.fr,NaN,2,Architecte sur le site faire.fr (2),https://ws-api.architectes.org/faire/v2/attest...,Tableau de l´Ordre,Architecte,Rénovation globale,cnoa,1,2019-06-18,2099-01-01
4,80976198400016,AT HOME,4 BOULEVARD GAETAN HERVE,35200,RENNES,48.090545,-1.671711,06 78 05 82 10,at-home-architecture@architectes.org,https://www.at-home-moe.com/,2,Architecte sur le site faire.fr (2),https://ws-api.architectes.org/faire/v2/attest...,Tableau de l´Ordre,Architecte,Rénovation globale,cnoa,1,2019-06-06,2099-01-01


## 3. Contrôle de la structure

In [4]:
print(df_rge.columns.tolist())
df_rge.info()

['siret', 'nom_entreprise', 'adresse', 'code_postal', 'commune', 'latitude', 'longitude', 'telephone', 'email', 'site_internet', 'code_qualification', 'nom_qualification', 'url_qualification', 'nom_certificat', 'domaine', 'meta_domaine', 'organisme', 'particulier', 'lien_date_debut', 'lien_date_fin']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158797 entries, 0 to 158796
Data columns (total 20 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   siret               158797 non-null  int64  
 1   nom_entreprise      158797 non-null  object 
 2   adresse             158797 non-null  object 
 3   code_postal         158797 non-null  int64  
 4   commune             158797 non-null  object 
 5   latitude            158728 non-null  float64
 6   longitude           158728 non-null  float64
 7   telephone           151591 non-null  object 
 8   email               156566 non-null  object 
 9   site_internet       103869 non-n

## 4. Filtrage du Val-d'Oise

Le code postal est converti en texte afin de pouvoir contrôler son format.

Le filtre conserve uniquement les codes :
- commençant par `95` ;
- comportant exactement **5 caractères**.

Ce second contrôle évite notamment d'intégrer par erreur des codes comme `9500`.

In [5]:
df_rge["code_postal"] = df_rge["code_postal"].astype(str)

df_rge_95 = df_rge[
    (df_rge["code_postal"].str.startswith("95")) &
    (df_rge["code_postal"].str.len() == 5)
].copy()

print("Dimensions Val-d'Oise :", df_rge_95.shape)
print(
    "Codes postaux de longueur différente de 5 :",
    (df_rge_95["code_postal"].str.len() != 5).sum()
)
print("Entreprises RGE distinctes :", df_rge_95["siret"].nunique())

Dimensions Val-d'Oise : (1607, 20)
Codes postaux de longueur différente de 5 : 0
Entreprises RGE distinctes : 553


## 5. Harmonisation des noms de communes

Les données RGE contiennent plusieurs variantes d'écriture pour une même commune : espaces, tirets, apostrophes, accents ou abréviations.

Le fichier DPE propre est utilisé **uniquement comme référentiel de noms de communes**.  
Une clé technique temporaire est créée pour rapprocher les libellés sans modifier directement les noms d'origine.


In [6]:
# Chargement uniquement des communes du fichier DPE propre
df_dpe_ref = pd.read_csv(
    "../Data/Output/dpe_val_doise_2024_2025_clean.csv",
    usecols=["commune"],
    encoding="utf-8-sig"
)

df_dpe_ref = df_dpe_ref.drop_duplicates().copy()

print("Communes de référence DPE :", df_dpe_ref["commune"].nunique())

Communes de référence DPE : 183


### Création d'une clé de comparaison

La clé sert uniquement au rapprochement :
- suppression des différences d'accents ;
- tirets et apostrophes remplacés par des espaces ;
- `St` harmonisé en `Saint` ;
- espaces multiples supprimés.

La colonne finale `commune` conservera ensuite le nom propre issu du référentiel DPE.


In [7]:
def creer_cle_commune(serie):
    return (
        serie
        .astype(str)
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.lower()
        .str.replace(r"[-']", " ", regex=True)
        .str.replace(r"\bst\b", "saint", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

df_rge_95["commune_cle"] = creer_cle_commune(df_rge_95["commune"])
df_dpe_ref["commune_cle"] = creer_cle_commune(df_dpe_ref["commune"])

print("Clés DPE uniques :", df_dpe_ref["commune_cle"].nunique())

Clés DPE uniques : 183


### Correction des dernières exceptions connues

Après normalisation, quelques variantes ne correspondent toujours pas exactement au référentiel DPE.  
Elles sont corrigées explicitement afin que le traitement soit reproductible après un redémarrage du kernel.

In [8]:
corrections_cles = {
    "arnouville les gonesse": "arnouville",
    "eragny sur oise": "eragny",
    "herblay": "herblay sur seine",
    "herblay sur seine": "herblay sur seine",
    "herblay sur sseine": "herblay sur seine",
    "isle adam": "l isle adam"
}

df_rge_95["commune_cle"] = (
    df_rge_95["commune_cle"]
    .replace(corrections_cles)
)

### Remplacement par les noms de référence DPE

Chaque clé RGE est maintenant remplacée par le libellé propre correspondant dans le fichier DPE.


In [9]:
ref_communes = (
    df_dpe_ref[["commune_cle", "commune"]]
    .drop_duplicates()
    .set_index("commune_cle")["commune"]
)

df_rge_95["commune"] = df_rge_95["commune_cle"].map(ref_communes)

print("Nombre de communes :", df_rge_95["commune"].nunique())
print("Communes non reconnues :", df_rge_95["commune"].isna().sum())

Nombre de communes : 102
Communes non reconnues : 0


### Contrôle attendu

Le résultat attendu pour le fichier actuel est :
- **102 communes**
- **0 valeur manquante dans `commune`**

La clé technique n'est plus nécessaire après ce contrôle.

In [10]:
# Suppression de la clé technique utilisée uniquement pour le rapprochement
df_rge_95 = df_rge_95.drop(columns=["commune_cle"])

## 6. Conversion des types

Les identifiants sont conservés comme texte.  
Les dates de début et de fin de qualification sont converties au format date.

In [11]:
df_rge_95["siret"] = df_rge_95["siret"].astype(str)
df_rge_95["code_postal"] = df_rge_95["code_postal"].astype(str)

df_rge_95["lien_date_debut"] = pd.to_datetime(
    df_rge_95["lien_date_debut"],
    errors="coerce"
)

df_rge_95["lien_date_fin"] = pd.to_datetime(
    df_rge_95["lien_date_fin"],
    errors="coerce"
)

print(
    df_rge_95[
        ["siret", "code_postal", "lien_date_debut", "lien_date_fin"]
    ].dtypes
)

print("\nDates devenues manquantes après conversion :")
print(
    df_rge_95[
        ["lien_date_debut", "lien_date_fin"]
    ].isna().sum()
)

siret                      object
code_postal                object
lien_date_debut    datetime64[ns]
lien_date_fin      datetime64[ns]
dtype: object

Dates devenues manquantes après conversion :
lien_date_debut    0
lien_date_fin      0
dtype: int64


## 7. Contrôle des doublons

In [12]:
print("Doublons exacts :", df_rge_95.duplicated().sum())

Doublons exacts : 0


## 8. Quelques contrôles analytiques

Une entreprise peut apparaître plusieurs fois lorsqu'elle possède plusieurs qualifications RGE.  
On distingue donc toujours **nombre de lignes / qualifications** et **nombre d'entreprises distinctes**.

In [13]:
print("Nombre de lignes :", len(df_rge_95))
print("Entreprises distinctes :", df_rge_95["siret"].nunique())
print("Communes distinctes :", df_rge_95["commune"].nunique())

print("\nRépartition par méta-domaine :")
print(df_rge_95["meta_domaine"].value_counts())

Nombre de lignes : 1607
Entreprises distinctes : 553
Communes distinctes : 102

Répartition par méta-domaine :
meta_domaine
Travaux d'efficacité énergétique          1079
Installations d'énergies renouvelables     400
Etudes énergétiques                         96
Rénovation globale                          31
Non renseigné                                1
Name: count, dtype: int64


### Entreprises RGE par commune

Le comptage utilise `nunique()` sur le SIRET afin de ne compter qu'une fois une entreprise possédant plusieurs qualifications.

In [14]:
entreprises_par_commune = (
    df_rge_95
    .groupby("commune")["siret"]
    .nunique()
    .sort_values(ascending=False)
)

entreprises_par_commune.head(10)

commune
Argenteuil                47
Sarcelles                 33
Herblay-sur-Seine         19
Cergy                     18
Saint-Ouen-l'Aumône       17
Eaubonne                  16
Deuil-la-Barre            16
Cormeilles-en-Parisis     14
Franconville              11
Saint-Brice-sous-Forêt    10
Name: siret, dtype: int64

## 9. Sélection des colonnes utiles

Les coordonnées géographiques et les coordonnées de contact restent disponibles dans le fichier brut, mais ne sont pas nécessaires à l'analyse SQL actuelle.

In [15]:
colonnes_utiles = [
    "siret",
    "nom_entreprise",
    "adresse",
    "code_postal",
    "commune",
    "code_qualification",
    "nom_qualification",
    "domaine",
    "meta_domaine",
    "organisme",
    "lien_date_debut",
    "lien_date_fin"
]

df_rge_clean = df_rge_95[colonnes_utiles].copy()

print("Dimensions du fichier final :", df_rge_clean.shape)
df_rge_clean.head()

Dimensions du fichier final : (1607, 12)


,siret,nom_entreprise,adresse,code_postal,commune,code_qualification,nom_qualification,domaine,meta_domaine,organisme,lien_date_debut,lien_date_fin
37,79118512700010,AGARAM,1 RUE DE LA FONTAINE DU GUE,95170,Deuil-la-Barre,2,Architecte sur le site faire.fr (2),Architecte,Rénovation globale,cnoa,2022-09-14,2099-01-01
65,82154955700014,MARIE LEGER ARCHITECTE,2 RUE JULES VINCENT,95410,Groslay,2,Architecte sur le site faire.fr (2),Architecte,Rénovation globale,cnoa,2019-09-06,2099-01-01
170,80105516100013,SARA ARCHITECTURE,39 BOULEVARD CLEMENCEAU,95220,Herblay-sur-Seine,2,Architecte sur le site faire.fr (2),Architecte,Rénovation globale,cnoa,2020-11-16,2099-01-01
339,79531109100016,HALLAK AND NASSER ARCHITECTS WORKSHOP,3 AVENUE MARECHAL JUIN,95500,Gonesse,2,Architecte sur le site faire.fr (2),Architecte,Rénovation globale,cnoa,2019-06-25,2099-01-01
447,30918116200038,CABINET D'ETUDES D'ARCHITECTURE ET D'URBANISME,5 RUE DE CHARS,95640,Marines,2,Architecte sur le site faire.fr (2),Architecte,Rénovation globale,cnoa,2019-09-06,2099-01-01


## 10. Nettoyage de caractères typographiques

Les guillemets typographiques sont remplacés par des guillemets standards afin d'éviter des problèmes lors de certains imports.

In [16]:
df_rge_clean = df_rge_clean.replace(
    {
        "“": '"',
        "”": '"'
    },
    regex=True
)

## 11. Contrôles finaux avant export

Ces contrôles permettent de vérifier que le fichier final est bien celui attendu avant de l'envoyer vers MySQL.


In [17]:
print("Nombre de lignes :", len(df_rge_clean))
print("Entreprises distinctes :", df_rge_clean["siret"].nunique())
print("Communes distinctes :", df_rge_clean["commune"].nunique())
print("Doublons exacts :", df_rge_clean.duplicated().sum())

print("\nValeurs manquantes :")
print(df_rge_clean.isna().sum())

Nombre de lignes : 1607
Entreprises distinctes : 553
Communes distinctes : 102
Doublons exacts : 0

Valeurs manquantes :
siret                 0
nom_entreprise        0
adresse               0
code_postal           0
commune               0
code_qualification    0
nom_qualification     0
domaine               0
meta_domaine          0
organisme             0
lien_date_debut       0
lien_date_fin         0
dtype: int64


## 12. Export du fichier propre

Le fichier est exporté en **UTF-8 standard**.  
On évite ici `utf-8-sig`, qui avait ajouté un caractère BOM au nom de la première colonne lors d'un import MySQL précédent.


In [18]:
print(
    "Nombre de libellés de communes après correction :",
    df_rge_95["commune"].nunique()
)

Nombre de libellés de communes après correction : 102


In [22]:
df_rge_clean.to_csv(
    "../Data/Output/rge_95_clean.csv",
    index=False,
    encoding="utf-8"
)

print("Export terminé : ../Data/Output/rge_val_doise_clean.csv")

Export terminé : ../Data/Output/rge_val_doise_clean.csv


In [23]:
# Vérification du fichier réellement exporté
df_test_rge = pd.read_csv(
    "../Data/Output/rge_95_clean.csv",
    dtype={
        "siret": str,
        "code_postal": str
    }
)

print("Dimensions :", df_test_rge.shape)
print("Entreprises :", df_test_rge["siret"].nunique())
print("Communes :", df_test_rge["commune"].nunique())
print("Communes manquantes :", df_test_rge["commune"].isna().sum())
print("Première colonne :", df_test_rge.columns[0])

Dimensions : (1607, 12)
Entreprises : 553
Communes : 102
Communes manquantes : 0
Première colonne : siret
